In [311]:
from functions import (load_json,
                       save_json,
                       past_example_components,
                       get_job_details,
                       get_prompts,
                       get_source_examples,
                       split_llm_outputs,
                       make_doc,
                       save_to_folder)
from gemini_chatbot import ChatBot

In [310]:
for k in [k for k in sys.modules if k=='functions']:
    sys.modules.pop(k)

In [ ]:
# Load details from past applications from json file, parse document components
past_examples = load_json('input/job_application_examples.json')
doc_components = past_example_components(past_examples)

# Get job posting details
job_details = get_job_details()

# Get prompt templates from .txt file
prompts = get_prompts()

In [50]:
# Initialize chat
prompt_intro = (prompts['init']
                   .replace('<ROLE>', job_details['role'])
                   .replace('<COMPANY>', job_details['company'])
                   .replace('<LOCATION>', job_details['location'])
                   .replace('<DETAILS>', job_details['job_posting']))
chatbot = ChatBot(prompt_intro)

In [ ]:
# Generate text for document components
raw_llm_outputs = {}
for key in [*doc_components[0].keys()]:
    prompt = ((prompts['r_job_bullets'] 
               if key in ['r_effo', 'r_dusa_vp', 'r_dusa_dir', 'r_dusa_sa', 'r_duk'] 
               else prompts[key])
              + '\n\nPREVIOUS EXAMPLES:\n' 
              + get_source_examples(doc_components, key))
    raw_llm_outputs[key] = chatbot.send(prompt)
    print(f'\rFinished {key}{' '*20}', end='')

In [320]:
# Extract feedback on how much new text was composed by AI, for QA
llm_outputs, llm_report = split_llm_outputs(raw_llm_outputs)

In [322]:
# Create resume and cover letter by inserting generated text into templates
resume = make_doc('resume', job_details, llm_outputs)
coverletter = make_doc('coverletter', job_details, llm_outputs)

In [323]:
# Write documents to files, update past examples json
past_examples_add = save_to_folder(job_details, resume, coverletter, llm_report)
past_examples += past_examples_add
save_json('input/job_application_examples.json', past_examples)